# Session 25 — AutoML-Based Smart Prediction System with Deployment

**Goal:** get the AutoML experience from Sessions 4 and 9 (search over models and
hyperparameters automatically) **without needing a cloud account** — using
[FLAML](https://microsoft.github.io/FLAML/) (Fast and Lightweight AutoML), a real,
open-source AutoML library that runs entirely on your own machine.

## Why this session exists alongside Sessions 4 and 9

Vertex AI AutoML and SageMaker Autopilot are managed services — powerful, but they
cost money per run and need a cloud account, which is why Sessions 4 and 9 were
reference-only in this sandbox. FLAML gives the same *idea* (automatic model
selection + hyperparameter search) as free, local, runnable code — a good default
when a managed service isn't available or justified for a smaller project.

## Prerequisites

```bash
pip install flaml
```
Runs entirely locally.

In [ ]:
import pandas as pd
import numpy as np
from flaml import AutoML
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

print("FLAML ready.")

## Step 1 — Load data and hand it to AutoML

Same heart disease dataset as the SageMaker/Vertex AI AutoML sessions, so results
are directly comparable in spirit (though not identical, since these are different
search spaces and libraries).

In [ ]:
df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
X, y = df.drop(columns="target"), df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"train: {len(X_train)}, test: {len(X_test)}")

## Step 2 — Run the AutoML search

`time_budget` caps wall-clock search time (the local equivalent of Vertex AI's
`budget_milli_node_hours` — but here it costs time, not money); `estimator_list`
tells FLAML which model families to consider.

In [ ]:
automl = AutoML()

automl.fit(
    X_train=X_train, y_train=y_train,
    task="classification",
    metric="roc_auc",
    time_budget=30,  # seconds -- short for a CodeAlong; increase for a real search
    estimator_list=["lgbm", "rf", "xgboost", "extra_tree", "lrl1"],
    verbose=0,
)

print(f"Best estimator: {automl.best_estimator}")
print(f"Best hyperparameters: {automl.best_config}")
print(f"Best CV score (1 - roc_auc): {automl.best_loss:.4f}")

## Step 3 — Evaluate on the held-out test set

In [ ]:
test_proba = automl.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_proba)
print(f"Test AUC: {test_auc:.4f}")

## Step 4 — Compare against a single fixed model

The value of AutoML in one number: how much better is "search over several model
families" than "commit to one reasonable default upfront"?

In [ ]:
from sklearn.ensemble import RandomForestClassifier

baseline = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train)
baseline_auc = roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1])

print(f"Fixed RandomForest(n_estimators=100) AUC: {baseline_auc:.4f}")
print(f"FLAML AutoML ({automl.best_estimator}) AUC:            {test_auc:.4f}")
print(f"Improvement: {test_auc - baseline_auc:+.4f}")

## Step 5 — Track the search with MLflow

FLAML doesn't include tracking itself — pair it with MLflow (Session 1) to keep a
permanent record of the search result, exactly like every other session's models.

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("session25-flaml-automl")

with mlflow.start_run(run_name=f"flaml_{automl.best_estimator}") as run:
    mlflow.log_param("best_estimator", automl.best_estimator)
    for k, v in automl.best_config.items():
        mlflow.log_param(f"config_{k}", v)
    mlflow.log_metric("test_auc", test_auc)
    mlflow.sklearn.log_model(automl.model.estimator, artifact_path="model")
    run_id = run.info.run_id
    print(f"Logged to MLflow: run_id={run_id}")

## Step 6 — Deploy the winning model behind an API

Same FastAPI pattern as every serving session — AutoML output is just another
scikit-learn-compatible model once the search is done.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI(title="Smart Prediction System (FLAML AutoML)")

class PatientFeatures(BaseModel):
    age: float; sex: float; cp: float; trestbps: float; chol: float
    fbs: float; restecg: float; thalach: float; exang: float
    oldpeak: float; slope: float; ca: float; thal: float

@app.post("/v1/predict")
def predict(features: PatientFeatures):
    row = np.array([[getattr(features, c) for c in X.columns]])
    proba = automl.predict_proba(row)[0, 1]
    return {"risk_probability": round(float(proba), 4), "model": automl.best_estimator}

client = TestClient(app)
sample = X_test.iloc[0].to_dict()
response = client.post("/v1/predict", json=sample)
print(response.status_code, response.json())

## What to try next

* Increase `time_budget` well beyond 30 seconds and watch `automl.best_loss` keep
  improving — FLAML's search is anytime, more budget generally means a better model.
* Add `starting_points=automl.best_config` for a warm-started re-search after new
  data arrives, instead of starting the hyperparameter search from scratch each time.
* Compare this local, free AutoML result against Session 4/9's managed cloud
  services on the same dataset if you have cloud access — the right tool depends on
  dataset size, budget, and whether you need the extra model families/scale a
  managed service offers.